# KUDS → RAZLOM-4 на GLM-5.1

Полный запуск: 1 вызов KUDS + 4 слепых предложения + 4 мутации + 4 независимые проверки = **13 запросов**.

Модель: `z-ai/glm-5.1` через OpenRouter. Для надёжного JSON reasoning отключён. Ответы сохраняются по стадиям, поэтому прерванный запуск можно продолжить без повторной оплаты успешных шагов.

In [ ]:
# Установка последней версии из GitHub
from pathlib import Path
import subprocess, sys

REPO = Path('/content/razlom4')
if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/marieabdlk-art/razlom4.git', str(REPO)
    ], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print('RAZLOM-4 установлен')

In [ ]:
# Ключ не сохраняется в ноутбуке и не выводится на экран
from getpass import getpass
import os

os.environ['OPENROUTER_API_KEY'] = getpass('OpenRouter API key: ')
assert os.environ['OPENROUTER_API_KEY'].strip(), 'Ключ не введён'
print('Ключ принят')

In [ ]:
# Измените задачу и критерии под себя
TASK_ID = 'new-idea-001'
QUESTION = 'Как создать новый механизм долгосрочной памяти для LLM-агента без бесконтрольного роста контекста?'
BASELINE = 'Сохранять историю, делать summary и извлекать релевантные фрагменты через vector search.'

HARD_CONSTRAINTS = [
    {'id': 'C1', 'text': 'Механизм не должен хранить весь исходный контекст бессрочно.'},
    {'id': 'C2', 'text': 'Должен существовать способ удалить ошибочную или устаревшую память.'},
    {'id': 'C3', 'text': 'Решение должно допускать измеримый эксперимент.'},
]

SUCCESS_TESTS = [
    {'id': 'T1', 'metric': 'качество на delayed-recall benchmark', 'threshold': 'не ниже baseline'},
    {'id': 'T2', 'metric': 'средний объём активного контекста', 'threshold': 'минимум на 40% ниже baseline'},
    {'id': 'T3', 'metric': 'удаление ложной памяти', 'threshold': 'не менее 95% проверочных случаев'},
]

contract = {
    'task_id': TASK_ID,
    'question': QUESTION,
    'baseline': BASELINE,
    'hard_constraints': HARD_CONSTRAINTS,
    'success_tests': SUCCESS_TESTS,
    'novelty_scope': 'panel',
}
contract

In [ ]:
# Предварительная оценка стоимости на 31 июля 2026
INPUT_PRICE = 0.966   # USD / 1M tokens, OpenRouter aggregate
OUTPUT_PRICE = 3.036  # USD / 1M tokens; reasoning входит в output

def estimate_cost(input_tokens, output_tokens):
    return (input_tokens * INPUT_PRICE + output_tokens * OUTPUT_PRICE) / 1_000_000

low = estimate_cost(40_000, 25_000)
expected = estimate_cost(60_000, 35_000)
high = estimate_cost(90_000, 50_000)
configured_ceiling = estimate_cost(150_000, 75_000)

print(f'Ожидаемый диапазон: ${low:.3f}–${high:.3f}')
print(f'Рабочая оценка:     ${expected:.3f}')
print(f'Грубый потолок при заполнении лимитов: ${configured_ceiling:.3f}')

In [ ]:
# Полный запуск — обычно занимает несколько минут
import json, time, sys
REPO = Path('/content/razlom4')
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from razlom4_full import FullPipeline, OpenRouterProvider, CheckpointProvider

MODEL = 'z-ai/glm-5.1'

class ProgressProvider(OpenRouterProvider):
    def complete_json(self, prompt, *, stage, temperature):
        print(f'→ {stage}', flush=True)
        result = super().complete_json(
            prompt, stage=stage, temperature=temperature
        )
        if self.usage_records:
            last = self.usage_records[-1]
            print(
                f"  {last['prompt_tokens']} input + "
                f"{last['completion_tokens']} output; "
                f"≈ ${last['estimated_cost_usd']:.4f}"
            )
        return result

base_provider = ProgressProvider(
    api_key=os.environ['OPENROUTER_API_KEY'],
    model=MODEL,
    reasoning_effort='none',
)
CHECKPOINT = Path('/content/razlom4_kuds_checkpoint.json')
provider = CheckpointProvider(
    base_provider,
    CHECKPOINT,
    allow_stale_prefixes=('review:',),
)
print('Checkpoint:', CHECKPOINT)

started = time.time()
result = FullPipeline(provider, n_candidates=12).run(contract)
elapsed = time.time() - started

OUTPUT = Path('/content/razlom4_kuds_glm51_result.json')
OUTPUT.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')

usage = result.get('usage', {})
actual_cost = usage.get('reported_cost')
if actual_cost is None:
    actual_cost = usage.get('estimated_cost_usd')

print('\n=== ГОТОВО ===')
print('Статус:', result['selection']['status'])
print('Победитель:', result['selection']['selected_candidate_id'])
print(f'Время: {elapsed / 60:.1f} мин')
print(f"Токены: {usage.get('prompt_tokens', 0)} input + {usage.get('completion_tokens', 0)} output")
print(f'Стоимость: ${actual_cost:.4f}' if actual_cost is not None else 'Стоимость недоступна')
print('Файл:', OUTPUT)

result['idea_dossier']

In [ ]:
# Скачать полный JSON с пулом идей, конфликтом, reviews и dossier
from google.colab import files
files.download(str(OUTPUT))